# Data source evaluations

Provide mapping from required variables -> data sources -> datasets.   Also include definitions for each column as defined in the zahan et al paper.  

## Target/Outcome Variables

| Variable | Type | Description | Source | Dataset |
|---|---|---|---|---|
| Vul_Count | Target | Number of publicly disclosed open security vulnerabilities within a package and its full dependency tree, including both direct and transitive dependencies | OpenSSF Scorecard Vulnerability metric (via OSV); denormalized from Scorecard score; raw counts retrieved separately for packages scoring 0 | df_pypi_scorecards.check_score (check_name=Vulnerabilities); OSV.raw_vulnerability_count (upstream source) |
| MTTR | Target | Mean time to remediate — average aggregated time a package uses outdated and vulnerable direct dependencies in its lifetime; quantifies how long vulnerable dependencies remain outdated after a fix version is released | Deps.dev (dependency/version/release data) + OSV (CVE data); algorithm from Rahman et al. [67]; available for ~15% of dataset (22,412 packages) | df_mttu_mttr.mttr |
| MTTU | Target | Mean time to update — average aggregated time a package uses outdated direct dependencies in its lifetime; quantifies how long dependencies remain outdated after any new version is released | Deps.dev (dependency/version/release data); algorithm from Rahman et al. [67]; available for full dataset (145,817 packages) | df_mttu_mttr.mttu |

## Scorecard Security Practice Predictors (Active in Zahan)

| Variable | Type | Description | Source | Dataset |
|---|---|---|---|---|
| Binary-Artifacts | Scorecard (Active) | Whether the project is free of checked-in binary files in its repository | OpenSSF Scorecard (GitHub repository analysis); scored 0–10 | df_pypi_scorecards.check_score (check_name=Binary-Artifacts) |
| Branch-Protection | Scorecard (Active) | Whether the project uses branch protection rules to safeguard important branches | OpenSSF Scorecard (GitHub repository analysis); scored 0–10 | df_pypi_scorecards.check_score (check_name=Branch-Protection) |
| Code-Review | Scorecard (Active) | Whether the project practices code review before code is merged into the repository | OpenSSF Scorecard (GitHub repository analysis); scored 0–10 | df_pypi_scorecards.check_score (check_name=Code-Review) |
| License | Scorecard (Active) | Whether the project declares a software license | OpenSSF Scorecard (GitHub repository analysis); scored 0–10 | df_pypi_scorecards.check_score (check_name=License) |
| Maintained | Scorecard (Active) | Whether the project is at least 90 days old and actively maintained | OpenSSF Scorecard (GitHub repository analysis); scored 0–10 | df_pypi_scorecards.check_score (check_name=Maintained) |
| Dependency-Update-Tool | Scorecard (Active) | Whether the project uses automated tools (e.g., Dependabot, Renovate) to help update its dependencies | OpenSSF Scorecard (GitHub repository analysis); scored 0–10 | df_pypi_scorecards.check_score (check_name=Dependency-Update-Tool) |
| Contributors | Scorecard (Active) | Whether the project has contributors from multiple different organizations, indicating diversity of contribution | OpenSSF Scorecard (GitHub repository analysis); scored 0–10 | df_pypi_scorecards.check_score (check_name=Contributors) |
| SAST | Scorecard (Active) | Whether the project uses static code analysis tools such as CodeQL to detect vulnerabilities | OpenSSF Scorecard (GitHub repository analysis); scored 0–10 | df_pypi_scorecards.check_score (check_name=SAST) |
| Security-Policy | Scorecard (Active) | Whether the project contains a documented security policy (e.g., SECURITY.md) for reporting vulnerabilities | OpenSSF Scorecard (GitHub repository analysis); scored 0–10 | df_pypi_scorecards.check_score (check_name=Security-Policy) |
| CI-Tests | Scorecard (Active) | Whether the project runs automated tests in CI pipelines (e.g., GitHub Actions, Prow) on pull requests; decomposed into binary presence indicator + effectiveness score due to >50% missing values (-1 indicates no pull requests found) | OpenSSF Scorecard (GitHub repository analysis); scored 0–10 | df_pypi_scorecards.check_score (check_name=CI-Tests) |
| Pinned-Dependencies | Scorecard (Active) | Whether the project declares and pins dependencies to specific versions to prevent unexpected updates; decomposed into binary presence indicator + effectiveness score due to >50% missing values (-1 indicates no declared dependencies found in GitHub) | OpenSSF Scorecard (GitHub repository analysis); scored 0–10 | df_pypi_scorecards.check_score (check_name=Pinned-Dependencies) |

## Scorecard Security Practice Predictors (Removed by Zahan)

| Variable | Type | Reason Removed | Source | Dataset |
|---|---|---|---|---|
| Token-Permissions | Scorecard (Removed) | High multicollinearity with Pinned-Dependencies (Spearman ≈ 1); removed at correlation analysis stage | OpenSSF Scorecard (GitHub repository analysis) | df_pypi_scorecards.check_score (check_name=Token-Permissions) |
| Dangerous-Workflow | Scorecard (Removed) | High multicollinearity with Pinned-Dependencies (Spearman ≈ 1); removed at correlation analysis stage | OpenSSF Scorecard (GitHub repository analysis) | df_pypi_scorecards.check_score (check_name=Dangerous-Workflow) |
| Fuzzing | Scorecard (Removed) | 99.8% of repositories scored 0, indicating near-zero adoption across the ecosystem | OpenSSF Scorecard (GitHub repository analysis) | df_pypi_scorecards.check_score (check_name=Fuzzing) |
| CII-Best-Practices | Scorecard (Removed) | 99.9% of repositories scored 0, indicating near-zero adoption across the ecosystem | OpenSSF Scorecard (GitHub repository analysis) | df_pypi_scorecards.check_score (check_name=CII-Best-Practices) |
| Signed-Releases | Scorecard (Removed) | ~98% of repositories received -1 (no GitHub release workflow present); practitioners use package registries rather than GitHub for releases | OpenSSF Scorecard (GitHub repository analysis) | df_pypi_scorecards.check_score (check_name=Signed-Releases) |
| Packaging | Scorecard (Removed) | ~98% of repositories received -1 (no CI/CD package release workflow on GitHub) | OpenSSF Scorecard (GitHub repository analysis) | df_pypi_scorecards.check_score (check_name=Packaging) |
| Vulnerabilities | Scorecard (Removed) | Used to construct Vul_Count (target variable); removed to avoid circularity | OpenSSF Scorecard (via OSV) | df_pypi_scorecards.check_score (check_name=Vulnerabilities) |

## Control Variables

| Variable | Type | Description | Source | Dataset |
|---|---|---|---|---|
| Contributors_CT | Control | Number of contributors to the GitHub repository; reflects team capacity and development activity; log-transformed due to high skewness | Libraries.io | df_libraries_io.contributions_count |
| Commit Staleness | Control | Time since the most recent commit; reflects recency of development activity and whether a project is actively maintained | GitHub REST API | df_feature_repo_age_staleness.commit_staleness_days |
| Download Count | Control | Number of package downloads in the past 12 months; reflects project adoption and external exposure/scrutiny; log-transformed due to high skewness | npm public API | df_pypi_file_downloads.total_downloads |
| Direct Dependencies | Control | Number of direct dependencies declared by the package; reflects external exposure and dependency complexity; log-transformed due to high skewness | Deps.dev | df_feature_dependencies_with_version.dependency_count; df_feature_dependencies_without_version.dependency_count |
| Repository Age | Control | Age of the GitHub repository in years; reflects project maturity, which may impact maintenance effort and security performance | GitHub REST API | df_feature_repo_age_staleness.repo_age_years |
| Repository Size | Control | Size of the GitHub repository; reflects project complexity and maintenance burden; log-transformed due to high skewness | Libraries.io | df_libraries_io.size |

In [1]:
from settings import load_settings

import sys
import polars as pl
from pathlib import Path

# Locate repo root by walking up until we find the src/ directory
_here = Path().resolve()
_repo_root = next(p for p in [_here, *_here.parents] if (p / "src").is_dir())
sys.path.insert(0, str(_repo_root / "src"))

settings = load_settings()

## Initial dataset (from deps.dev)

In [2]:
df_initial_dataset = pl.read_parquet(f"{settings.initial_dataset_path}/*.parquet")


In [3]:
df_initial_dataset.shape

(209889019, 8)

In [4]:
df_initial_dataset.columns

['SnapshotAt',
 'package_name',
 'package_version',
 'package_published_at',
 'dep_name',
 'dep_version',
 'MinimumDepth',
 'github_repo']

In [5]:
df_initial_dataset.head()


SnapshotAt,package_name,package_version,package_published_at,dep_name,dep_version,MinimumDepth,github_repo
datetime[μs],str,str,datetime[μs],str,str,i64,str
2023-04-10 21:01:49.219309,"""12factor-vault""","""0.1.23""",null,"""certifi""","""2022.12.7""",3,"""jdelic/12factor-vault"""
2023-04-10 21:01:49.219309,"""12factor-vault""","""0.1.23""",null,"""charset-normalizer""","""3.1.0""",3,"""jdelic/12factor-vault"""
2023-04-10 21:01:49.219309,"""12factor-vault""","""0.1.23""",null,"""django-dbconn-retry""","""0.1.7""",1,"""jdelic/12factor-vault"""
2023-04-10 21:01:49.219309,"""12factor-vault""","""0.1.23""",null,"""hvac""","""1.1.0""",1,"""jdelic/12factor-vault"""
2023-04-10 21:01:49.219309,"""12factor-vault""","""0.1.23""",null,"""idna""","""3.4.0""",3,"""jdelic/12factor-vault"""


## Libraries.io dataset

In [6]:
# load libraries.io dataset
df_libraries_io = pl.read_parquet(f"{settings.librariesio_path}/*.parquet")


In [7]:
df_libraries_io.shape

(18149, 29)

In [8]:
df_libraries_io.columns

['full_name',
 'description',
 'fork',
 'created_at',
 'updated_at',
 'pushed_at',
 'size',
 'stargazers_count',
 'language',
 'has_issues',
 'has_wiki',
 'has_pages',
 'forks_count',
 'open_issues_count',
 'default_branch',
 'subscribers_count',
 'license',
 'contributions_count',
 'has_readme',
 'has_changelog',
 'has_contributing',
 'has_license',
 'has_coc',
 'has_threat_model',
 'has_audit',
 'status',
 'rank',
 'github_contributions_count',
 'github_id']

In [9]:
df_libraries_io.head()

full_name,description,fork,created_at,updated_at,pushed_at,size,stargazers_count,language,has_issues,has_wiki,has_pages,forks_count,open_issues_count,default_branch,subscribers_count,license,contributions_count,has_readme,has_changelog,has_contributing,has_license,has_coc,has_threat_model,has_audit,status,rank,github_contributions_count,github_id
str,str,bool,str,str,str,i64,i64,str,bool,bool,bool,i64,i64,str,i64,str,i64,str,str,str,str,str,str,str,str,i64,i64,str
"""pelican-plugins/image-process""","""Pelican plugin that automates …",false,"""2015-02-06T18:51:48.000Z""","""2025-11-18T08:51:02.632Z""","""2025-11-18T08:50:08.000Z""",10840,58,"""Python""",true,false,false,22,4,"""main""",5,"""AGPL-3.0""",18,"""README.md""","""CHANGELOG.md""","""CONTRIBUTING.md""","""LICENSE""",null,null,null,null,7,18,"""30427628"""
"""lucidrains/rotary-embedding-to…","""Implementation of Rotary Embed…",false,"""2021-06-29T19:05:39.000Z""","""2026-02-08T08:51:21.815Z""","""2025-07-27T01:25:42.000Z""",113,716,"""Python""",true,true,false,63,18,"""main""",11,"""MIT""",5,"""README.md""",null,null,"""LICENSE""",null,null,null,null,14,5,"""381470350"""
"""soulmachine/delimited-protobuf""","""A read/write library for lengt…",false,"""2021-09-23T21:49:18.000Z""","""2021-09-24T23:00:49.034Z""","""2021-09-24T21:16:11.000Z""",13,0,"""Python""",true,true,false,0,0,"""main""",1,"""Apache-2.0""",1,"""README.md""",null,null,"""LICENSE""",null,null,null,null,3,1,"""409752898"""
"""canonical/craft-store""","""Python API to communicate with…",false,"""2021-08-06T11:59:22.000Z""","""2025-10-22T20:17:12.664Z""","""2025-10-21T19:59:15.000Z""",1294,5,"""Python""",true,false,false,19,21,"""main""",5,"""LGPL-3.0""",14,"""README.md""",null,"""CONTRIBUTING.md""","""LICENSE""",null,null,null,null,6,14,"""393363282"""
"""flaskbb/flask-debugtoolbar-war…","""Adds warnings support to flask…",false,"""2018-06-04T03:40:08.000Z""","""2025-01-18T03:14:57.702Z""","""2018-06-13T00:46:42.000Z""",19,2,"""Python""",true,true,false,1,0,"""master""",3,"""MIT""",1,"""README.rst""","""CHANGELOG""",null,"""LICENSE""",null,null,null,null,3,1,"""135967074"""


## mttu_mttr

**data pending ongoing work - may change**

In [10]:
df_mttu_mttr = pl.read_parquet(f"{settings.mttu_mttr_path}/*.parquet")

In [11]:
df_mttu_mttr.shape

(150, 6)

In [12]:
df_mttu_mttr.columns

['package_name',
 'github_repo',
 'min_SnapshotDate',
 'max_SnapshotDate',
 'mttr',
 'mttu']

In [13]:
df_mttu_mttr.head()

package_name,github_repo,min_SnapshotDate,max_SnapshotDate,mttr,mttu
str,str,date,date,f64,f64
"""devicetree""","""zephyrproject-rtos/python-devi…",2026-02-23,2026-02-23,0.0,43.335404
"""flask-oidc2""","""vishnu667/flask-oidc2""",2026-02-23,2026-02-23,0.0,25.469323
"""jaws-libp""","""jeffersonlab/jaws-libp""",2026-02-23,2026-02-23,5.997493,40.102629
"""microhmm""","""howl-anderson/microhmm""",2026-02-23,2026-02-23,0.0,4.71409
"""spacecomx-cdk-billing-alarm""","""spacecomx/cdk-billing-alarm""",2026-02-23,2026-02-23,0.0,0.287747


## OSV

This data is not directly used as it is also included in deps.dev and scorecard data.   It is indirectly used in the MTTU/MTTR calcluations and is optional.  It is stored in our data structure only for performance of the MTTU/MTTR calcs.

## PyPI file downloads


In [14]:
# load pypi file downloads
df_pypi_file_downloads = pl.read_parquet(f"{settings.feature_total_downloads_path}/*.parquet")


In [15]:
df_pypi_file_downloads.shape

(9160855, 3)

In [16]:
df_pypi_file_downloads.columns


['package_name', 'package_version', 'total_downloads']

In [17]:
df_pypi_file_downloads.head()

package_name,package_version,total_downloads
str,str,i64
"""python-dateutil""","""2.9.0.post0""",7484189911
"""six""","""1.17.0""",6918414978
"""idna""","""3.10""",5191430067
"""typing-extensions""","""4.15.0""",5153700743
"""packaging""","""25.0""",5108442748


## OSSF Scorecards 

In [18]:
# load pypi scorecards
df_pypi_scorecards = pl.read_parquet(f"{settings.pypi_scorecards_path}/*.parquet")


In [19]:
df_pypi_scorecards.shape

(36977913, 7)

In [20]:
df_pypi_scorecards.columns


['scorecard_date',
 'repo_name',
 'repo_commit',
 'scorecard_version',
 'aggregate_score',
 'check_name',
 'check_score']

In [21]:
df_pypi_scorecards.head()

scorecard_date,repo_name,repo_commit,scorecard_version,aggregate_score,check_name,check_score
date,str,str,str,f64,str,i64
2023-10-30,"""github.com/keybase/pykeybasebo…","""38c84aac41fc03d4d2693fb316288d…","""v4.13.1-26-g478f347e""",2.5,"""Code-Review""",6
2023-10-30,"""github.com/keybase/pykeybasebo…","""38c84aac41fc03d4d2693fb316288d…","""v4.13.1-26-g478f347e""",2.5,"""Pinned-Dependencies""",-1
2023-10-30,"""github.com/paweltroka/signalr-…","""effa7347016174e460d3c3e3f2a9d0…","""v4.13.1-26-g478f347e""",2.5,"""Binary-Artifacts""",10
2023-10-30,"""github.com/kaixuyang/lunarcale…","""888497987ffe7fecca94f544c03e56…","""v4.13.1-26-g478f347e""",3.0,"""Dangerous-Workflow""",-1
2023-10-30,"""github.com/jjmontesl/codenamiz…","""5af5f8bca3b39d18806a051fd112ea…","""v4.13.1-26-g478f347e""",3.0,"""CII-Best-Practices""",0


In [22]:
# determine unique repos by scorecard_date, ordered descending
df_pypi_scorecards.group_by("scorecard_date").agg(
    pl.n_unique("repo_name").alias("repo_count")
).sort("scorecard_date", descending=True)

scorecard_date,repo_count
date,u32
2026-02-23,19725
2026-02-16,19718
2026-02-09,19714
2026-01-26,19775
2026-01-19,19779
…,…
2023-05-08,19791
2023-05-01,19863
2023-04-24,19863


In [23]:
# from df_pypi_scorecards, where scorecard_date is '2026-02-23' extract repo_name, aggregate_score, check_name (as columns), check_score (as value for each of the check_name columns)
df_pypi_scorecards.filter(pl.col("scorecard_date") == pl.date(2026, 2, 23)).pivot(
    index=["scorecard_date", "repo_name", "aggregate_score"],
    columns="check_name",
    values="check_score"
)

/tmp/ipykernel_96853/782165201.py:2: DeprecationWarning: the argument `columns` for `DataFrame.pivot` is deprecated. It was renamed to `on` in version 1.0.0.
  df_pypi_scorecards.filter(pl.col("scorecard_date") == pl.date(2026, 2, 23)).pivot(


scorecard_date,repo_name,aggregate_score,Pinned-Dependencies,Packaging,License,Dangerous-Workflow,Fuzzing,Token-Permissions,SAST,Maintained,Signed-Releases,Branch-Protection,CII-Best-Practices,Binary-Artifacts,Security-Policy,Code-Review
date,str,f64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64
2026-02-23,"""github.com/tastatham/geobootst…",1.5,-1,-1,0,-1,0,-1,0,0,-1,0,0,10,0,0
2026-02-23,"""github.com/gnaneshwar441/busin…",1.5,-1,-1,0,-1,0,-1,0,0,-1,0,0,10,0,0
2026-02-23,"""github.com/platonnetwork/plato…",1.8,0,-1,10,-1,0,-1,0,0,-1,0,0,10,0,0
2026-02-23,"""github.com/alashkov83/s_dbw""",1.9,-1,-1,10,-1,0,-1,0,0,0,0,0,10,0,1
2026-02-23,"""github.com/ncaptier/stabilized…",1.9,-1,-1,10,-1,0,-1,0,0,-1,0,0,9,0,0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
2026-02-23,"""github.com/csenshi/validator""",3.4,0,-1,10,10,0,0,0,0,-1,-1,0,10,0,3
2026-02-23,"""github.com/ska-sa/codex-africa…",2.9,0,10,9,0,0,0,0,5,-1,-1,0,10,0,2
2026-02-23,"""github.com/komuw/sewer""",2.9,0,-1,10,10,0,0,0,0,-1,0,0,10,0,1


In [24]:
# show individual check_name values from the pypi scorecards
df_pypi_scorecards.select("check_name").unique().to_pandas().head(50)

,check_name
0,Vulnerabilities
1,Token-Permissions
2,Dangerous-Workflow
3,Signed-Releases
4,Code-Review
5,Pinned-Dependencies
6,Branch-Protection
7,Binary-Artifacts
8,Security-Policy
9,Packaging


## Augmented - unique packages

Contains listing of package + repo combinations used as a filter source from the large dataset in initial dataset in order to extract information from other sources

In [25]:
# load unique packages dataset 
df_unique_packages = pl.read_parquet(f"{settings.unique_packages_path}")

In [26]:
df_unique_packages.shape

(42447, 4)

In [27]:
df_unique_packages.columns

['package_name', 'github_repo', 'min_snapshot_date', 'max_snapshot_date']

In [28]:
df_unique_packages.head()

package_name,github_repo,min_snapshot_date,max_snapshot_date
str,str,datetime[μs],datetime[μs]
"""python-geotiepoints""","""github.com/pytroll/python-geot…",2023-04-10 21:01:49.219309,2026-02-23 21:00:56.375217
"""polyaxon-client""","""github.com/polyaxon/polyaxon-c…",2023-04-10 21:01:49.219309,2026-02-23 21:00:56.375217
"""smartapi-python""","""github.com/angelbroking-github…",2023-07-03 21:01:30.072045,2026-02-23 21:00:56.375217
"""flake8-secure-coding-standard""","""github.com/takishima/flake8-se…",2023-04-10 21:01:49.219309,2026-02-23 21:00:56.375217
"""sbo-selenium""","""github.com/safarijv/sbo-seleni…",2023-04-10 21:01:49.219309,2026-02-23 21:00:56.375217


## Augmented - Features - dep count - with version

In [29]:
# load feature dependency data with version
df_feature_dependencies_with_version = pl.read_parquet(f"{settings.feature_dependency_count_with_version_path}")

In [30]:
df_feature_dependencies_with_version.shape

(144253, 3)

In [31]:
df_feature_dependencies_with_version.columns

['package_name', 'package_version', 'dependency_count']

In [32]:
df_feature_dependencies_with_version.head()

package_name,package_version,dependency_count
str,str,u32
"""numpy""","""2.4.2""",9187
"""numpy""","""2.3.5""",8394
"""numpy""","""2.3.4""",8062
"""numpy""","""2.3.3""",7893
"""numpy""","""2.4.1""",7789


## Augmented - Features - dep count without version

In [33]:
# load feature dependency counts without version 
df_feature_dependencies_without_version = pl.read_parquet(f"{settings.feature_dependency_count_without_version_path}")


In [34]:
df_feature_dependencies_without_version.shape


(22768, 2)

In [35]:
df_feature_dependencies_without_version.columns


['package_name', 'dependency_count']

In [36]:
df_feature_dependencies_without_version.head()

package_name,dependency_count
str,u32
"""numpy""",10939
"""requests""",6883
"""pandas""",5244
"""scipy""",4627
"""matplotlib""",3531


## Augmented - Features - repo age and staleness

In [37]:
## load feature repo age and staleness
df_feature_repo_age_staleness = pl.read_parquet(f"{settings.feature_repo_age_and_staleness_path}")


In [38]:
df_feature_repo_age_staleness.shape

(42447, 4)

In [39]:
df_feature_repo_age_staleness.columns

['package_name', 'github_repo', 'repo_age_years', 'commit_staleness_days']

In [40]:
df_feature_repo_age_staleness.head()

package_name,github_repo,repo_age_years,commit_staleness_days
str,str,f64,i64
"""tabicl""","""soda-inria/tabicl""",1.1,0
"""django-tagulous""","""radiac/django-tagulous""",10.85,266
"""requests-oidc""","""tsweeney-dust/requests-oidc""",3.18,1026
"""interop""","""illumina/interop""",10.14,93
"""aiocaldav""","""thomaschiroux/aiocaldav""",7.75,2789


## Unique Packages

In [41]:
# load Unique_packages.parquet
unique_packages = pl.read_parquet(settings.unique_packages_path)


In [42]:
unique_packages.shape

(42447, 4)

In [43]:
unique_packages.head()

package_name,github_repo,min_snapshot_date,max_snapshot_date
str,str,datetime[μs],datetime[μs]
"""python-geotiepoints""","""github.com/pytroll/python-geot…",2023-04-10 21:01:49.219309,2026-02-23 21:00:56.375217
"""polyaxon-client""","""github.com/polyaxon/polyaxon-c…",2023-04-10 21:01:49.219309,2026-02-23 21:00:56.375217
"""smartapi-python""","""github.com/angelbroking-github…",2023-07-03 21:01:30.072045,2026-02-23 21:00:56.375217
"""flake8-secure-coding-standard""","""github.com/takishima/flake8-se…",2023-04-10 21:01:49.219309,2026-02-23 21:00:56.375217
"""sbo-selenium""","""github.com/safarijv/sbo-seleni…",2023-04-10 21:01:49.219309,2026-02-23 21:00:56.375217


## Augmented - Features - Librariesio

In [44]:
# load libraries.io features
df_libraries_io_features = pl.read_parquet(settings.feature_libraries_io_path)

In [45]:
df_libraries_io.shape

(18149, 29)

In [46]:
df_libraries_io_features.head()

full_name,contributions_cnt,size
str,i64,i64
"""pelican-plugins/image-process""",18,10840
"""lucidrains/rotary-embedding-to…",5,113
"""soulmachine/delimited-protobuf""",1,13
"""canonical/craft-store""",14,1294
"""flaskbb/flask-debugtoolbar-war…",1,19


## Augmented - features - pypi downloads

In [47]:
# load PyPI downloads features
df_pypi_downloads_features = pl.read_parquet(settings.feature_pypi_downloads_path)
df_pypi_downloads_features.head()

package_name,package_version,total_downloads
str,str,i64
"""python-dateutil""","""2.9.0.post0""",7484189911
"""six""","""1.17.0""",6918414978
"""idna""","""3.10""",5191430067
"""typing-extensions""","""4.15.0""",5153700743
"""packaging""","""25.0""",5108442748


## Augmented - features - ossf scorecard

In [48]:
# load OSSF Scorecard features
df_ossf_scorecard = pl.read_parquet(settings.feature_ossf_scorecard_path)

In [49]:
df_ossf_scorecard.shape

(19786, 18)

In [50]:
df_ossf_scorecard.head()

scorecard_date,repo_name,aggregate_score,Packaging,Token-Permissions,SAST,Fuzzing,Maintained,Branch-Protection,Dangerous-Workflow,Binary-Artifacts,Vulnerabilities,Security-Policy,Signed-Releases,License,Code-Review,CII-Best-Practices,Pinned-Dependencies
date,str,f64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64
2024-01-01,"""moveaxlab/sdk-boilerplate-py""",2.5,-1,-1,0,0,0,0,-1,10,9,0,-1,0,0,0,-1
2024-01-01,"""idin/pensieve""",3.0,-1,-1,0,0,0,0,-1,10,10,0,-1,9,0,0,-1
2024-01-01,"""fhpythonutils/pylsr""",3.0,-1,-1,0,0,0,0,-1,10,10,4,0,10,0,0,-1
2024-01-01,"""allonkleinlab/cospar""",3.0,-1,0,0,0,0,0,10,10,5,0,-1,10,0,0,0
2024-01-01,"""mkuranowski/osmiter""",3.0,-1,-1,0,0,0,0,-1,10,10,0,-1,10,0,0,-1


## TODO 

Initial dataset - leave as is
(original from Bigquery Dataset)
(no publish to features - used as primary reference for list of repos and dependencies)


libraries.io dataset
(original extracted from libraries.io API)
- DONE clean up gathering code in source
- publish filtered set to features


mttr_mttu
- clean gathering code in source
- publish filtered set to features


OSV - no change - may be able to delete - confirm with Emily after mttr/u complete.

PyPI file downloads
- (originalk from BigQuery dataset sql)
- DONE publish filtered set to features


OSSF scorecards
(original from BigQuery Dataset)
- DONE publish filtered set to features
